# 1. Data Processing¶

- Install and import libriaries
- Load the dataset
- Preview the data
- Format and standardize the data.

In [0]:
dbutils.library.restartPython()
%pip install xgboost
%pip install us

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Importing pandas for data analysis and manipulation (handling data frames)
import pandas as pd

# Used for array manipulation and matrix operations
import numpy as np

# Importing matplotlib for creating visualizations
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.cm as cm

# Import seaborn for statistical data visualization
import seaborn as sns

# Use choropleth maps 
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Import MinMaxScaler to normalize feature to 0-1 range
from sklearn.preprocessing import MinMaxScaler

from xgboost import XGBClassifier
from xgboost import XGBRegressor
from xgboost import plot_importance

from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

from scipy.stats import ttest_ind

# Import the 'us' library to access U.S. state information (names, abbreviations, FIPS codes, etc.)
import us

# To suppress warnings
import warnings
warnings.filterwarnings("ignore")

In [0]:
# Read dataset from Databricks Catalog
realtor_data_spark = spark.read.table("workspace.default.realtor_data_zip")
df = realtor_data_spark.toPandas()

print("Raw data:")
display(df.head())
display(df.info())

print("\n")

# Convert it to a pandas DataFrame for downstream EDA and modeling
def load_data():
    return spark.read.table("workspace.default.realtor_data_zip").toPandas()
    
# Load the dataset
df = load_data()

# Convert numeric columns to integers, even if it contains messy or non-numeric values
cols_to_int = ['brokered_by', 'price', 'bed', 'bath', 'street', 'zip_code', 'house_size']
for col in cols_to_int:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    
# Fill the string-type columns with "Unknown" where values are NaN 
cols_to_fill = ['status', 'city', 'state']
df[cols_to_fill] = df[cols_to_fill].fillna("Unknown")

# Convert to datetime
df['prev_sold_date'] = pd.to_datetime(df['prev_sold_date'], errors='coerce')

# Fill missing values with placeholder
placeholder_date = pd.to_datetime("1900-01-01")
plot_df = df[df['prev_sold_date'] != placeholder_date]

# Fill NaN values in the float-type column with 0.0
df['acre_lot'] = df['acre_lot'].fillna(0.0)

print("Data standardization:")
display(df.head())
display(df.iloc[163364:163388])
display(df.iloc[[530530]])

Raw data:


brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,null
52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,null
103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,null
31239.0,for_sale,145000.0,4.0,2.0,0.1,1947675.0,Ponce,Puerto Rico,731.0,1800.0,null
34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,null,null


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 12 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   brokered_by     1047537 non-null  float64
 1   status          1048575 non-null  object 
 2   price           1047485 non-null  float64
 3   bed             738569 non-null   float64
 4   bath            723607 non-null   float64
 5   acre_lot        870247 non-null   float64
 6   street          1041799 non-null  float64
 7   city            1047715 non-null  object 
 8   state           1048568 non-null  object 
 9   zip_code        1048416 non-null  float64
 10  house_size      683689 non-null   float64
 11  prev_sold_date  497886 non-null   object 
dtypes: float64(8), object(4)
memory usage: 96.0+ MB


Data standardization:


brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
103378,for_sale,105000,3,2,0.12,1962661,Adjuntas,Puerto Rico,601,920,null
52707,for_sale,80000,4,2,0.08,1902874,Adjuntas,Puerto Rico,601,1527,null
103379,for_sale,67000,2,1,0.15,1404990,Juana Diaz,Puerto Rico,795,748,null
31239,for_sale,145000,4,2,0.1,1947675,Ponce,Puerto Rico,731,1800,null
34632,for_sale,65000,6,2,0.05,331151,Mayaguez,Puerto Rico,680,0,null


brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
79270,for_sale,144500,2,2,0.08,336787,Pittsburgh,Pennsylvania,15216,0,2014-11-20T00:00:00.000Z
19512,for_sale,330000,4,3,0.37,727797,Elizabeth,Pennsylvania,15037,0,2007-10-03T00:00:00.000Z
22611,for_sale,259900,3,2,0.5,1050836,Canonsburg,Pennsylvania,15317,1560,2014-07-23T00:00:00.000Z
22611,for_sale,204900,3,2,0.0,1082904,Homestead,Pennsylvania,15120,0,2021-01-06T00:00:00.000Z
10705,for_sale,148000,0,0,0.09,236400,Pittsburgh,Pennsylvania,15227,0,2002-04-29T00:00:00.000Z
51500,for_sale,39900,2,1,0.01,539121,McKeesport,Pennsylvania,15132,0,2004-10-29T00:00:00.000Z
30750,for_sale,299900,4,3,0.1,213868,Pittsburgh,Pennsylvania,15216,1888,1987-05-20T00:00:00.000Z
83644,for_sale,269900,3,2,0.41,158722,Canonsburg,Pennsylvania,15317,1262,2017-04-06T00:00:00.000Z
22611,for_sale,270000,3,3,0.18,714188,Pittsburgh,Pennsylvania,15226,1440,2018-06-22T00:00:00.000Z
110141,for_sale,179900,0,0,0.06,930393,Pittsburgh,Pennsylvania,15227,0,2021-08-05T00:00:00.000Z


brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
11761,ready_to_build,204990,3,0,0.0,828031,Binder,Unknown,96999,1273,null


For further Metropolitan Statistical Area (MSA) analysis, related datasets were merged with consistent data types to ensure smooth processing.

zipcode_FIPS_cbsa_crosswalk_2015.csv includes the following columns:
- zipcode
- FIPS
- CountyName
- cbsacode
- cbsatitle
- metromicro

OMB_cbsa_2015.csv provides supplementary information and includes:
- CBSA Code
- CBSA Title
- Metropolitan/Micropolitan Statistical Area

To fill in the missing cbsatitle and metromicro fields in zipcode_FIPS_cbsa_crosswalk_2015.csv, a merge was performed using the cbsacode as the key.

Challenge: Dataset Alignment and Cross-Framework Integration

To successfully merge multiple datasets, it was critical to ensure proper alignment of shared dimensions (e.g., CBSA code) across sources.

One of the main challenges was handling messy official data, where column headers were not standardized and appeared in non-default rows. This required programmatically realigning headers and cleaning key fields before performing joins.

Additionally, the workflow needed to seamlessly switch between Spark DataFrames and pandas DataFrames depending on the task.

This hybrid approach ensured both scalability and analytical flexibility throughout the pipeline.

In [0]:
# Load data
crosswalk  = spark.read.table("workspace.default.zipcode_FIPS_cbsa_crosswalk_2015").toPandas()
cbsa_names  = spark.read.table("workspace.default.OMB_cbsa_2015").toPandas()

# Find the real header (The row 1, index=1)
cbsa_names.columns = cbsa_names.iloc[1]

# Drop the useless columns at the beginning.
cbsa_names = cbsa_names.iloc[3:].reset_index(drop=True)

print(cbsa_names.columns)

Index(['CBSA Code', 'Metropolitan Division Code', 'CSA Code', 'CBSA Title',
       'Metropolitan/Micropolitan Statistical Area',
       'Metropolitan Division Title', 'CSA Title', 'County/County Equivalent',
       'State Name', 'FIPS State Code', 'FIPS County Code',
       'Central/Outlying County'],
      dtype='object', name=1)


In [0]:
# Rename columns in cbsa_names dataframe to shorter, consistent lowercase names
cbsa_names = cbsa_names.rename(columns={
    'CBSA Code': 'cbsacode',
    'CBSA Title': 'cbsatitle',
    'Metropolitan/Micropolitan Statistical Area': 'metromicro'
})

# Ensure data type consistency
crosswalk['cbsacode'] = crosswalk['cbsacode'].astype(str).str.strip()
cbsa_names['cbsacode'] = cbsa_names['cbsacode'].astype(str).str.strip()

# To remedy the special value '99999', mark it as unknown
crosswalk.loc[crosswalk['cbsacode'] == '99999', ['cbsatitle', 'metromicro']] = ['No CBSA', 'Unknown']

# Merge crosswalk and cbsa_names, and do a left merge based on cbsacode (mainly crosswalk)
crosswalk_full = pd.merge(crosswalk, cbsa_names, on='cbsacode', how='left', suffixes=('_orig', ''))

In [0]:
# First make sure df['zip_code'] is a 5-code string format
df['zip_code'] = df['zip_code'].astype(str).str.zfill(5).str.strip()

# Make sure crosswalk_full['zipcode'] is in the same format
crosswalk_full['zipcode'] = crosswalk_full['zipcode'].astype(str).str.zfill(5).str.strip()

# Merge with df, and match zip_code to crosswalk_full’s zipcode
df_merged = df.merge(
    crosswalk_full[['zipcode', 'cbsatitle', 'metromicro']],
    left_on='zip_code',
    right_on='zipcode',
    how='left'
).drop(columns=['zipcode'])  # If no need the 'zipcode' field after merging, delete it

# Check the output
print(df_merged[['zip_code', 'cbsatitle', 'metromicro']].head(10))

  zip_code     cbsatitle                     metromicro
0    00601  Adjuntas, PR  Micropolitan Statistical Area
1    00601  Adjuntas, PR  Micropolitan Statistical Area
2    00795     Ponce, PR  Metropolitan Statistical Area
3    00795     Ponce, PR  Metropolitan Statistical Area
4    00795     Ponce, PR  Metropolitan Statistical Area
5    00795     Ponce, PR  Metropolitan Statistical Area
6    00795     Ponce, PR  Metropolitan Statistical Area
7    00795     Ponce, PR  Metropolitan Statistical Area
8    00795     Ponce, PR  Metropolitan Statistical Area
9    00731     Ponce, PR  Metropolitan Statistical Area
